> **Historical research track.** This notebook documents the earlier trajectory-anomaly experiment. It is preserved as reproducible evidence and does not define the current SADAR Analyst Console product.

# Phase 4 EDA — Dataset #6 (in-flight emergencies) LEMD inspection

**Purpose.** Inspect OpenSky scientific Dataset #6 — the 832 flights that squawked **7700** (general emergency), 2018-01-01 to 2020-01-29 (Olive et al., *OpenSky Report 2020*) — to scope it as the **Layer 4 external-validation** set (D-008). We count the LEMD-associated subset, tabulate emergency categories, and decide how finely Layer 4 can slice.

**⚠️ FIREWALL.** This is *inspection only* (metadata: counts, categories, airports). It is Phase-4-legal. **Scoring these flights with the model is SEALED until Phase 7** (after model selection per D-006). Do NOT load these into any training/validation/threshold/selection step. Same posture as the test set.

**Cross-references**
- `docs/research/trajectory-anomaly/lifecycle/dataset6-emergency-external-validation.md` — the full exploration / acquisition spec (read this first; §9 has the authoritative trajectory-radius result).
- `docs/research/trajectory-anomaly/lifecycle/decisions/D-008-output-validation-layers.md` — the 5-layer validation stack; this is Layer 4.
- `docs/research/trajectory-anomaly/lifecycle/decisions/D-011-real-derived-synthetic-anomalies.md` — uses the non-LEMD flights as injection sources; firewall split.
- `docs/research/trajectory-anomaly/lifecycle/07-eval-prep.md` > "Layer 4" — the Phase-7 execution protocol + pre-committed finding template.
- `docs/research/trajectory-anomaly/original-design.md` > "External validation" — the narrative slot this fills.

**Environment note (2026-06-01).** The `traffic` library (dep, v2.13) is currently **broken in the backend venv** — it imports `DatetimeTZBlock` from a pandas internal path that newer pandas removed (`ImportError`). So `from traffic.data.datasets import squawk7700` fails today. This notebook works around it by reading the dataset's **metadata CSV directly from Zenodo** (record `3937483`), which needs only `pandas`. Fixing `traffic` (pin pandas / bump traffic) is a team-level dependency decision — see the env note at the bottom.

In [ ]:
import pandas as pd
from pathlib import Path

LEMD = "LEMD"  # Madrid-Barajas ICAO
# 200 km radius around LEMD (same window as cycle 3) — used in the trajectory pass (§4 / doc §9).
LEMD_REF = (40.4936, -3.5668)

# Local cache of the Zenodo metadata file (gitignored data/ dir).
CACHE = Path("../data/external/squawk7700_metadata.csv")
ZENODO_META = "https://zenodo.org/api/records/3937483/files/squawk7700_metadata.csv/content"

## 1. Load the metadata

If the cache is missing, download it once (Zenodo blocks default user-agents — send a browser UA). The 131 MB `squawk7700_trajectories.parquet.gz` is **not** needed here; it is only required for the trajectory pass (§4) and the sealed Phase-7 scoring.

In [ ]:
if not CACHE.exists():
    import urllib.request
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(ZENODO_META, headers={"User-Agent": "Mozilla/5.0"})
    CACHE.write_bytes(urllib.request.urlopen(req, timeout=120).read())
    print("downloaded ->", CACHE)

md = pd.read_csv(CACHE)
print("total 7700 flights (global):", len(md))
print("columns:", list(md.columns))

## 2. LEMD-association by airport code (first cut only)

We associate by **airport ICAO code** (`origin` / `destination` / `landing` / `diverted` == LEMD), **not** by Filter B/D. Filter B/D select for *normal LEMD operation geometry*; applying them to emergencies would discard exactly the trajectory-anomalous flights Layer 4 exists to test (circular). See the exploration doc §3.

**⚠️ This is only a first cut.** The trajectory pass (§4) shows airport-code is *not* the right Layer-4 filter — Dataset #6 stores emergency-window segments, so most airport-code-LEMD flights have no trajectory near LEMD.

In [ ]:
airport_cols = [c for c in ["origin", "destination", "landing", "diverted"] if c in md.columns]
mask = pd.Series(False, index=md.index)
for c in airport_cols:
    mask |= (md[c] == LEMD)
sub = md[mask].copy()

print(f"LEMD-associated flights (any of {airport_cols} == LEMD): {len(sub)}")
for c in airport_cols:
    print(f"  {c}==LEMD: {int((md[c]==LEMD).sum())}")

In [ ]:
show = [c for c in ["flight_id", "callsign", "typecode", "origin", "destination",
                     "landing", "diverted", "avh_problem"] if c in md.columns]
sub[show]

In [ ]:
# Relationship to LEMD (non-exclusive) + emergency category + aircraft type
arr = (sub["landing"] == LEMD) | (sub["destination"] == LEMD)
dep = (sub["origin"] == LEMD)
div = (sub["diverted"] == LEMD)
print("arriving/landing LEMD:", int(arr.sum()))
print("departing LEMD      :", int(dep.sum()))
print("diverted TO LEMD    :", int(div.sum()))
print("\navh_problem (LEMD subset):")
print(sub["avh_problem"].value_counts(dropna=False).to_string())
print("\ntypecode (LEMD subset):")
print(sub["typecode"].value_counts(dropna=False).to_string())

## 3. Airport-code findings (run 2026-06-01)

**N = 6** LEMD-associated 7700 flights by airport code (of 832 global):

| flight_id | type | origin | dest | landing | diverted | avh_problem |
|---|---|---|---|---|---|---|
| IBK6241_20180616 | B738 | BIKF | LEMD | EGBB | EGBB | hydraulics |
| LAN706_20180918 | B789 | SCEL | LEMD | — | — | — |
| EZY42LB_20181213 | A319 | EGPH | LEMD | EGGD | EGGD | — |
| AFR11DN_20190816 | A321 | LFPG | LEMD | — | LFBD | cabin_pressure |
| BCS63A_20190829 | A306 | LEMD | EHAM | LEMD | LEMD | — |
| LCO1501_20191208 | B763 | LEMD | GVAC | — | — | — |

**But this count is misleading for Layer 4 — see §4.** Airport code says where a flight was *intended* to go; it does not say the flight flew *near* LEMD during the emergency window that Dataset #6 actually stores.

## 4. Trajectory-radius pass — the authoritative Layer-4 filter

> **⚠️ Correction (2026-06-01): the N=6 airport-code count above is superseded for Layer-4 purposes.** The radius pass on the trajectory parquet (haversine to `LEMD_REF`, all 832 flights) showed Dataset #6 stores *emergency-window segments*, so **5 of the 6 airport-code-LEMD flights have zero trajectory points within 200 km of LEMD** (their emergency happened en route / at diversion, hundreds–thousands of km away).

Of the airport-code 6, only **BCS63A** (min 2.9 km, departed LEMD and returned — a **turn-back**, 1,239 pts <50 km) is actually near LEMD. The authoritative scoreable set is the trajectory radius:

| radius | flights |
|---|---|
| < 10 km | 1 |
| < 30 km | 2 |
| < 50 km | 4 |
| < 100 km | 7 |
| < 200 km | 7 |

**Scoreable Layer-4 set ≈ 7 within 200 km; ~4 reach the TMA (<50 km); only BCS63A is close-in.** The three TMA-transit recoveries: `JAF20K_20190919` (16.9 km), `TOM517_20190705` (34.4 km, engine, diverted LFBD), `BLX236_20180201` (39.6 km). Full table + consequences in `docs/research/trajectory-anomaly/lifecycle/dataset6-emergency-external-validation.md §9`; firewall split refined in **D-011**.

_(The radius pass code reads the 131 MB trajectory parquet; it is documented in doc §9 rather than re-run here so this notebook stays metadata-only and fast.)_

## 5. Implications and next steps

**This is the "settle once N is known" decision (exploration doc §8):**

1. **N is tiny — ~7 scoreable, 1 close-in.** Layer 4 is **per-flight case studies + Mann-Whitney**, not AUROC. **BCS63A is the qualitative centerpiece** (a real LEMD turn-back the model never saw).
2. **The Western-Europe fallback (07-eval-prep Layer-4) is now the likely path** for any statistical power — widen to 7700 flights within 200 km of *any* major Western-European hub, with the documented conflation caveat.
3. **N=7 is too small to stratify by emergency category** (§5 of the exploration doc); fold category into the qualitative per-flight notes.
4. **Runway-projection concern is concrete:** only BCS63A is close enough for the LEMD-anchored projection to be meaningful; the 50–200 km transit flights sit at the edge of the model's spatial domain (likely trivially high for "wrong place", not "emergency").

**Phase 6:** leave Dataset #6 untouched. One-line reminder at the Phase 6 entry gate.

**Phase 7 (sealed until then):** resample 1 s → 10 s, run the locked model through the *same* preprocessing + train scaler, score the in-range set once, report pooled percentile + Mann-Whitney U + per-flight qualitative notes (BCS63A first); fill `[N]`/`[X]`/`[K]` in writeup 09.

---

### Environment fix needed (team decision)

`traffic` 2.13 fails to import in the backend venv (`ImportError: cannot import name 'DatetimeTZBlock' from 'pandas.core.internals.blocks'`) — a pandas-version incompatibility. Any notebook importing `traffic` is blocked until this is resolved (pin pandas to a `traffic`-compatible version, or bump `traffic`). Given the container pins versions and `uv.lock` is intentionally uncommitted, raise this with the team rather than churning deps locally. The metadata-direct path used here sidesteps it for Phase-4 inspection; the Phase-7 trajectory pass reads the parquet directly (also fine without `traffic`).